# P(r) Inversion Demo

Model-free pair distance distribution analysis with `sans_fitter.pr_inversion`:

- Simulate a protein-like sphere dataset and load it with `data_ops.load()`
- Explore D_max **before** trusting any inversion — every result is conditional on it
- Run `auto_invert()` (automatic selection of `n_terms` and `alpha`) and read the diagnostics
- Plot P(r) with its uncertainty band and the fit against the data
- Export the result to CSV

P(r) inversion is model-free: no model setup is involved anywhere. It works
directly on datasets — `data_ops.load(...)` results or `fitter.data`.

## 1. Simulate and load sphere data

A sphere of radius R = 60 Å with 2% noise. The sphere form factor is analytic —
`I(q) = I(0)·[3(sin(u) − u·cos(u))/u³]²` with `u = qR` — so the simulation
needs nothing beyond numpy, and the oracle values are known exactly:
D_max = 2R = 120 Å, Rg = √(3/5)·R ≈ 46.5 Å, I(0) = 100.

With measured data you would skip the simulation and start at
`data_ops.load('your_file.csv')`.

The data is simulated background-free, so we use `fit_background=False`
throughout — the normal workflow for buffer-subtracted protein data
(a fitted flat background can absorb I(0) and bias Rg).

In [ ]:
import numpy as np

from sans_fitter import data_ops, pr_inversion

radius = 60.0
i0_true = 100.0
q = np.geomspace(0.005, 0.25, 80)
u = q * radius
i_clean = i0_true * (3.0 * (np.sin(u) - u * np.cos(u)) / u**3) ** 2

rng = np.random.default_rng(seed=42)
sigma = 0.02 * i_clean
i_noisy = i_clean + rng.normal(0.0, sigma)

with open('simulated_protein.csv', 'w') as f:
    f.write('Q,I,dI\n')
    for q_value, i_value, di_value in zip(q, i_noisy, sigma):
        f.write(f'{q_value},{i_value},{di_value}\n')

data = data_ops.load('simulated_protein.csv')

FIT_BACKGROUND = False
print(f'{len(data.x)} points, Q = [{data.x.min():.3g}, {data.x.max():.3g}]')

## 2. Explore D_max first

We start from a **deliberately wrong** guess (100 Å) and scan a wide window.
The chi-squared minimum and the Rg/I(0) plateau locate the true D_max.

In [ ]:
scan = pr_inversion.explore_dmax(
    data, d_max=100.0, dmin=80.0, dmax=160.0, n_points=17,
    fit_background=FIT_BACKGROUND,
)
best_dmax = float(scan.d_max_values[np.argmin(scan.data_chisq)])
print(f'Chi-squared minimum at D_max = {best_dmax:.1f} Å (true value: {2 * radius:.0f} Å)')
scan.plot(quantity='all')

## 3. Automatic inversion

`auto_invert()` estimates the number of basis terms and the regularization
constant, then inverts. The chosen values are recorded on the result
(`result.n_terms`, `result.alpha`); `format_summary()` collects every
diagnostic, including the positivity fractions (the fit is unconstrained, so
P(r) *can* go negative — unlike GNOM/ATSAS).

In [ ]:
result = pr_inversion.auto_invert(data, d_max=best_dmax, fit_background=FIT_BACKGROUND)
print(result.format_summary())
print(f'\nExpected sphere Rg = {np.sqrt(3 / 5) * radius:.2f} Å, I(0) = {i0_true:.1f}')

## 4. Plots

P(r) with its 1σ band, then the fit against the measured data with residuals
(the dataset is passed explicitly — the result stores the model, not the data).

In [ ]:
result.plot_pr()

In [ ]:
result.plot_fit(data)

## 5. Explicit control and export

The estimators can also be called directly — `estimate_n_terms` returns the
authoritative `(n_terms, alpha)` pair evaluated during its scan.

In [ ]:
estimate = pr_inversion.estimate_n_terms(data, d_max=best_dmax,
                                         fit_background=FIT_BACKGROUND)
print(estimate.message)

result = pr_inversion.invert(
    data, d_max=best_dmax,
    n_terms=estimate.n_terms, alpha=estimate.alpha,
    fit_background=FIT_BACKGROUND,
)
result.save_csv('pr_result.csv')
print("Saved P(r) curve and diagnostics to 'pr_result.csv'")

## Things to know

- **Q range is honoured**: `fitter.set_q_range()` restricts the inversion
  exactly like it restricts fits.
- **Missing dI** triggers fabricated uncertainties (with a warning and an
  `uncertainties_fabricated` flag) — chi-squared diagnostics are then not
  interpretable.
- **Shannon limits are checked**: warnings fire when `d_max > π/q_min` or
  `n_terms` exceeds the channel count `ceil(q_max·d_max/π)`.
- **`regularizer='sasview'`** reproduces SasView's exact smoothing operator
  for comparison; the default `'corrected'` penalizes the true second
  derivative on a resolved quadrature grid.
- Slit smearing (USANS) is not supported; pinhole dQ resolution is ignored,
  as in SasView.